In [9]:
from collections import deque
import numpy as np
class FifoDelayLine:
    def __init__(self, delay_samples):
        self.buffer = deque([0] * delay_samples, maxlen=delay_samples)

    def push_pop(self, sample):
        """
        Pushes new sample and pops the oldest 
        1. The oldest sample (at the left) is popped.
        2. The new sample is appended to the right.
        Because `maxlen` is set, the deque automatically discards the leftmost
        element when a new element is added, but we pop it first to return it.
        """
        if self.buffer.maxlen == 0:
            return sample # No delay, 
        # For a full deque, append will discard the leftmost item.
        # We need to capture it before it's gone.
        oldest_sample = self.buffer[0]
        self.buffer.append(sample)
        return oldest_sample

def load_all_mic_data():
    all_data = {}
    # Mapping: mic0->40, mic1->46, mic2->52, mic3->58
    mic_mapping = {
        'mic0': 40,
        'mic1': 46,
        'mic2': 52,
        'mic3': 58
    }
    for mic_name, mic_id in mic_mapping.items():
        mic_data = []
        for bram_idx in range(4):
            filename = f'./data/bram_nodelay_{mic_name}_{mic_id}_brams{bram_idx}.npy'
            data = np.load(filename)
            mic_data.append(data)
        all_data[mic_name] = {
            'id': mic_id,
            'brams': np.array(mic_data)
        }
    return all_data

def beamform_sum(delayed_samples):
    sample_sum = np.sum(delayed_samples)
    return np.clip(sample_sum, -32768, 32767)
# double power = 0.0;
# //       for (uint32_t sample = 0; sample < mic_size; sample++) {
# //         power += beamformed_signal[sample] * beamformed_signal[sample];
# //       }

In [10]:
import numpy as np
import matplotlib.pyplot as plt


def apply_delays_using_shift(raw_data, delays):
    num_mics, num_samples = raw_data.shape
    delayed_data = np.zeros_like(raw_data)
    for mic_idx in range(num_mics):
        delayed_data[mic_idx] = apply_delay_simple(raw_data[mic_idx], delays[mic_idx])
    return delayed_data


def process_audio_shift(raw_audio_data, delays):
    num_samples = raw_audio_data.shape[1]
    beamformed_output = np.zeros(num_samples)
    delayed_samples = []
    delayed_data = apply_delays_using_shift(raw_audio_data, delays)
    for s in range(num_samples):
        delayed_samples = delayed_data[:, s]
        assert len(delayed_samples) == 4, "Expected 4 delayed samples"
        sum = beamform_sum(delayed_samples)
        beamformed_output[s] = sum 
        
    return beamformed_output


def apply_delay_simple(signal, delay_samples):
    if delay_samples == 0:
        return signal.copy()
    delayed = np.zeros_like(signal)
    delayed[delay_samples:] = signal[:-delay_samples]
    
    return delayed


In [11]:
num_microphones = 4
DELAYS_MIC0_SOURCE = [0, 5, 8, 4]  # delays in samples for BRAM0, BRAM1, BRAM2, BRAM3
DELAYS_MIC1_SOURCE = [5,0,4,8]  # delays in samples for BRAM0, BRAM1, BRAM2, BRAM3
DELAYS_MIC2_SOURCE = [8,4,0,5]  # delays in samples for BRAM0, BRAM1, BRAM2, BRAM3
DELAYS_MIC3_SOURCE = [4,8,5,0]  # delays in samples for BRAM0, BRAM1, BRAM2, BRAM3
raw_data = load_all_mic_data()
beam0 = process_audio_shift(raw_data['mic0']['brams'], DELAYS_MIC0_SOURCE)
beam1 = process_audio_shift(raw_data['mic1']['brams'], DELAYS_MIC1_SOURCE)
beam2 = process_audio_shift(raw_data['mic2']['brams'], DELAYS_MIC2_SOURCE)
beam3 = process_audio_shift(raw_data['mic3']['brams'], DELAYS_MIC3_SOURCE)
beams = [beam0, beam1, beam2, beam3]
power =[]
for i, beam in enumerate(beams):
    p = np.sum(beam ** 2)
    power.append(p)
    print(f'Beamformed Output Mic{i} Source Power: {p}')    
reference_mic_idx = np.argmin(np.array(power))
print(f'Reference microphone index (lowest power): Mic{reference_mic_idx}')
reference_mic_idx = np.argmax(np.array(power))
print(f'Reference microphone index (highest power): Mic{reference_mic_idx}')

# for i, beam in enumerate(beams):
#     power = np.sum(beam ** 2)
#     plt.figure(figsize=(10, 4))
#     plt.plot(beam, label=f'Beamformed Output Mic{i} Source')
#     plt.title(f'Beamformed Output for Mic{i} as Source {power}')
#     plt.xlabel('Sample Index')
#     plt.ylabel('Amplitude')
#     plt.legend()
#     plt.grid()
#     plt.show()


Beamformed Output Mic0 Source Power: 282171564752.0
Beamformed Output Mic1 Source Power: 110467414908.0
Beamformed Output Mic2 Source Power: 159459585512.0
Beamformed Output Mic3 Source Power: 48699118108.0
Reference microphone index (lowest power): Mic3
Reference microphone index (highest power): Mic0


In [12]:
import numpy as np
import matplotlib.pyplot as plt

def process_audio_fifo(raw_audio_data, fifos):
    num_samples = raw_audio_data.shape[1]
    beamformed_output = np.zeros(num_samples)
    for s in range(num_samples):
        raw_samples_at_t = raw_audio_data[:, s]
        # print(raw_samples_at_t)
        assert len(raw_samples_at_t) == 4, "Expected 4 microphone channels"
        delayed_samples = apply_delays_with_fifo(raw_samples_at_t, fifos)
        sum = beamform_sum(delayed_samples)
        beamformed_output[s] = sum 
        
    return beamformed_output

def apply_delays_with_fifo(raw_samples_at_t, fifos):
    num_microphones = 4
    delayed_samples = np.zeros(num_microphones)
    for c in range(num_microphones):
        # For each channel, push the new sample and get the delayed one
        delayed_samples[c] = fifos[c].push_pop(raw_samples_at_t[c])
    return delayed_samples

In [13]:
num_microphones = 4
DELAYS_MIC0_SOURCE = [0, 5, 8, 4]  # delays in samples for BRAM0, BRAM1, BRAM2, BRAM3
fifos0 = [FifoDelayLine(delay) for delay in DELAYS_MIC0_SOURCE]

DELAYS_MIC1_SOURCE = [5,0,4,8]  # delays in samples for BRAM0, BRAM1, BRAM2, BRAM3
fifos1 = [FifoDelayLine(delay) for delay in DELAYS_MIC1_SOURCE]

DELAYS_MIC2_SOURCE = [8,4,0,5]  # delays in samples for BRAM0, BRAM1, BRAM2, BRAM3
fifos2 = [FifoDelayLine(delay) for delay in DELAYS_MIC2_SOURCE]

DELAYS_MIC3_SOURCE = [4,8,5,0]  # delays in samples for BRAM0, BRAM1, BRAM2, BRAM3
fifos3 = [FifoDelayLine(delay) for delay in DELAYS_MIC3_SOURCE]
raw_data = load_all_mic_data()
print("Loaded raw data for all microphones.", raw_data.keys())
# print(raw_data['mic0']['brams'].shape)
 
beam0 = process_audio_fifo(raw_data['mic0']['brams'], fifos0)
beam1 = process_audio_fifo(raw_data['mic1']['brams'], fifos1)
beam2 = process_audio_fifo(raw_data['mic2']['brams'], fifos2)
beam3 = process_audio_fifo(raw_data['mic3']['brams'], fifos3)
beams = [beam0, beam1, beam2, beam3]
power =[]
for i, beam in enumerate(beams):
    p = np.sum(beam ** 2)
    power.append(p)
    print(f'Beamformed Output Mic{i} Source Power: {p}')    

reference_mic_idx = np.argmin(np.array(power))
print(f'Reference microphone index (lowest power): Mic{reference_mic_idx}')
reference_mic_idx = np.argmax(np.array(power))
print(f'Reference microphone index (highest power): Mic{reference_mic_idx}')

# for i, beam in enumerate(beams):
#     power = np.sum(beam ** 2)
#     plt.figure(figsize=(10, 4))
#     plt.plot(beam, label=f'Beamformed Output Mic{i} Source')
#     plt.title(f'Beamformed Output for Mic{i} as Source {power}')
#     plt.xlabel('Sample Index')
#     plt.ylabel('Amplitude')
#     plt.legend()
#     plt.grid()
#     plt.show()


Loaded raw data for all microphones. dict_keys(['mic0', 'mic1', 'mic2', 'mic3'])
Beamformed Output Mic0 Source Power: 282171564752.0
Beamformed Output Mic1 Source Power: 110467414908.0
Beamformed Output Mic2 Source Power: 159459585512.0
Beamformed Output Mic3 Source Power: 48699118108.0
Reference microphone index (lowest power): Mic3
Reference microphone index (highest power): Mic0
